# Thai Sign Language - Curriculum Train (All Data → TSL-51)
Phase 1: pretrain on all 1938 examples (tsl51+thaisignvis+youtube_sl25_thai), early-stop on val_loss.  
Phase 2: finetune on TSL-51 only (252 examples), early-stop on val_chrf.  
Output: runtime-ready export in /kaggle/working/phase2.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

def find_dataset(slug: str) -> str:
    candidates = [
        f'/kaggle/input/{slug}',
        f'/kaggle/input/datasets/orbitorls/{slug}',
    ]
    for candidate in candidates:
        if os.path.isdir(candidate):
            return candidate
    return candidates[0]

CODE_DATASET = find_dataset('thai-sign-code')
MIXED_DATA_RAW = find_dataset('thai-sign-mixed-all-v6-archived')
TSL51_DATA_RAW = find_dataset('thai-sign-tsl51')
MT5_LOCAL = find_dataset('thai-sign-mt5small')
SEED_DIR = find_dataset('thai-sign-tsl51-seed')

PHASE1_DIR = '/kaggle/working/phase1'
PHASE2_DIR = '/kaggle/working/phase2'
CODE = '/tmp/thai_sign_code'

os.makedirs(PHASE1_DIR, exist_ok=True)
os.makedirs(PHASE2_DIR, exist_ok=True)
shutil.rmtree(CODE, ignore_errors=True)
os.makedirs(CODE, exist_ok=True)

# Unpack code bundle
repo_bundle = os.path.join(CODE_DATASET, 'repo_bundle.zip')
if os.path.isfile(repo_bundle):
    with zipfile.ZipFile(repo_bundle) as archive:
        archive.extractall(CODE)
else:
    for name in ['config.py', 'requirements.txt']:
        src = os.path.join(CODE_DATASET, name)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(CODE, name))
    for name in ['src', 'scripts']:
        src_dir = os.path.join(CODE_DATASET, name)
        if os.path.isdir(src_dir):
            shutil.copytree(src_dir, os.path.join(CODE, name), dirs_exist_ok=True)
    for name in ['src.zip', 'scripts.zip']:
        archive_path = os.path.join(CODE_DATASET, name)
        if os.path.isfile(archive_path):
            with zipfile.ZipFile(archive_path) as archive:
                archive.extractall(CODE)

# GPU check — allow P100 (sm_60).
# This Kaggle account consistently receives P100 regardless of requested accelerator.
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,compute_cap,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True, timeout=30,
)
if result.returncode != 0:
    raise RuntimeError('No GPU detected in this Kaggle session.')
gpu_name, gpu_cap, gpu_mem = [p.strip() for p in result.stdout.strip().split('\n')[0].split(',')]
allow_legacy_sm60 = True
is_legacy_gpu = float(gpu_cap) < 7.0
print(f'[setup] GPU: {gpu_name} (sm_{gpu_cap.replace(".", "")}) mem={gpu_mem} | allow_legacy_sm60={allow_legacy_sm60}')

if is_legacy_gpu and not allow_legacy_sm60:
    raise RuntimeError(f'Rejecting unsupported GPU sm_{gpu_cap.replace(".", "")} — request a different accelerator.')

# Install base dependencies (no torch upgrade yet — test existing torch first)
import sys as _sys, subprocess as _sp
for package in ['transformers>=4.40', 'sentencepiece>=0.2.0', 'sacrebleu>=2.4.0', 'portalocker>=2.0', 'pandas>=2.0']:
    _sp.check_call([_sys.executable, '-m', 'pip', 'install', '-q', package])

# CUDA sanity test: verify the pre-installed torch can actually run ops on this GPU.
# P100 (sm_60) is dropped from PyTorch cu121+ prebuilt wheels but cu118 still includes it.
import torch as _torch
print(f'[setup] Pre-installed torch={_torch.__version__} cuda_available={_torch.cuda.is_available()}')
_cuda_ok = False
if _torch.cuda.is_available():
    try:
        _t = _torch.zeros(4, 4, device='cuda')
        _s = (_t + 1).sum().item()
        _cuda_ok = True
        print(f'[setup] Pre-installed torch CUDA works (sum={_s})')
    except Exception as _e:
        print(f'[setup] Pre-installed torch CUDA failed: {_e}')

if not _cuda_ok and is_legacy_gpu and allow_legacy_sm60:
    # torch 2.1.x cu121 still includes sm_60 kernels and has Python 3.12 wheels
    # (sm_60 was dropped in torch 2.2+; torch 2.0.1 lacks Python 3.12 wheels)
    print('[setup] Installing PyTorch 2.1.2+cu121 (sm_60-compatible, Python 3.12 support)...')
    try:
        _sp.check_call([
            _sys.executable, '-m', 'pip', 'install', '-q',
            'torch==2.1.2', 'torchvision==0.16.2', 'torchaudio==2.1.2',
            '--index-url', 'https://download.pytorch.org/whl/cu121',
        ])
    except Exception as _e_cu121:
        print(f'[setup] cu121 install failed ({_e_cu121}), trying cu118 fallback...')
        _sp.check_call([
            _sys.executable, '-m', 'pip', 'install', '-q',
            'torch==2.1.2', 'torchvision==0.16.2', 'torchaudio==2.1.2',
            '--index-url', 'https://download.pytorch.org/whl/cu118',
        ])
    # Reload torch after reinstall
    import importlib
    import torch as _t2
    importlib.reload(_t2)
    import torch as _torch
    print(f'[setup] Reloaded torch={_torch.__version__}')
    try:
        _t = _torch.zeros(4, 4, device='cuda')
        _s = (_t + 1).sum().item()
        _cuda_ok = True
        print(f'[setup] cu118 torch CUDA works (sum={_s})')
    except Exception as _e:
        raise RuntimeError(f'CUDA unavailable on {gpu_name} (sm_{gpu_cap}) even with cu118: {_e}')
elif not _cuda_ok:
    raise RuntimeError(f'CUDA not available and GPU is not sm_60 — cannot proceed without GPU')

# Unpack mixed-all features.zip → writable working dir (if present)
features_archive = os.path.join(MIXED_DATA_RAW, 'features.zip')
if os.path.isfile(features_archive):
    mixed_runtime = Path('/kaggle/working/datasets') / Path(MIXED_DATA_RAW).name
    shutil.rmtree(mixed_runtime, ignore_errors=True)
    mixed_runtime.mkdir(parents=True, exist_ok=True)
    for name in ['manifest.csv', 'manifest_quality.json']:
        src = os.path.join(MIXED_DATA_RAW, name)
        if os.path.isfile(src):
            shutil.copy2(src, mixed_runtime / name)
    feature_root = mixed_runtime / 'features'
    feature_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(features_archive) as archive:
        archive.extractall(feature_root)
    MIXED_DATA = str(mixed_runtime)
else:
    # Individual .npy files already unpacked in Kaggle input mount
    MIXED_DATA = MIXED_DATA_RAW

# TSL-51 data (read-only mount, no unpack needed)
TSL51_DATA = TSL51_DATA_RAW

# Seed Phase 1 with best_model_state.pt from the chrF-87 incumbent
seed_state = os.path.join(SEED_DIR, 'best_model_state.pt')
seed_size = os.path.getsize(seed_state) if os.path.isfile(seed_state) else 0
if seed_size > 1_000_000:
    dst = os.path.join(PHASE1_DIR, 'best_model_state.pt')
    if not os.path.isfile(dst):
        shutil.copy2(seed_state, dst)
        print(f'[setup] Seeded Phase 1 from {seed_state} ({seed_size/1e9:.2f} GB)')
    PHASE1_RESUME_ARGS = ['--resume', 'best_state']
else:
    print(f'[setup] WARNING: seed not found or empty at {seed_state} ({seed_size} bytes) — Phase 1 starts cold')
    PHASE1_RESUME_ARGS = []

base_model = MT5_LOCAL if os.path.isdir(MT5_LOCAL) else 'google/mt5-small'

print({
    'torch': _torch.__version__, 'gpu': gpu_name, 'gpu_cap': gpu_cap, 'gpu_mem': gpu_mem,
    'mixed_data': MIXED_DATA, 'tsl51_data': TSL51_DATA,
    'phase1_dir': PHASE1_DIR, 'phase2_dir': PHASE2_DIR,
    'seed_dir': SEED_DIR, 'base_model': base_model,
    'phase1_resume_args': PHASE1_RESUME_ARGS,
})

In [ ]:
import os, shutil
from pathlib import Path
import pandas as pd
import torch

def validate_dataset(data_root: str, label: str) -> str:
    """Validate manifest + npy paths; returns the (possibly patched) data_root."""
    manifest_path = os.path.join(data_root, 'manifest.csv')
    if not os.path.isfile(manifest_path):
        raise FileNotFoundError(f'[{label}] manifest.csv not found: {manifest_path}')
    df = pd.read_csv(manifest_path)
    if 'npy_path' not in df.columns:
        raise ValueError(f'[{label}] npy_path column missing from {manifest_path}')
    features_dir = os.path.join(data_root, 'features')
    sample_paths = [os.path.join(data_root, str(v)) for v in df['npy_path'].head(min(8, len(df)))]
    missing_paths = [p for p in sample_paths if not os.path.isfile(p)]
    if missing_paths and os.path.isdir(features_dir):
        patched_dir = Path('/kaggle/working/datasets') / f'{Path(data_root).name}_manifestfix'
        shutil.rmtree(patched_dir, ignore_errors=True)
        patched_dir.mkdir(parents=True, exist_ok=True)
        for name in ['manifest_quality.json']:
            src = os.path.join(data_root, name)
            if os.path.isfile(src):
                shutil.copy2(src, patched_dir / name)
        df['npy_path'] = [
            str((Path(features_dir) / str(v)).resolve())
            if os.path.isfile(os.path.join(features_dir, str(v))) else str(v)
            for v in df['npy_path']
        ]
        manifest_out = patched_dir / 'manifest.csv'
        df.to_csv(manifest_out, index=False)
        data_root = str(patched_dir)
        sample_paths = [str(v) if os.path.isabs(str(v)) else os.path.join(data_root, str(v))
                        for v in df['npy_path'].head(min(8, len(df)))]
        missing_paths = [p for p in sample_paths if not os.path.isfile(p)]
    if missing_paths:
        raise FileNotFoundError({'label': label, 'data_root': data_root, 'missing': missing_paths})
    print(f'[{label}] rows={len(df)}, sources={df["source"].value_counts().to_dict() if "source" in df.columns else "n/a"}')
    return data_root

MIXED_DATA = validate_dataset(MIXED_DATA, 'mixed')
TSL51_DATA = validate_dataset(TSL51_DATA, 'tsl51')

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
# === Phase 1: pretrain on ALL data (tsl51 + thaisignvis + youtube_sl25_thai = 1938 examples) ===
import os, subprocess, sys

train_script_candidates = [
    os.path.join(CODE, 'scripts', 'kaggle_train_pose_t5.py'),
    os.path.join(CODE, 'kaggle_train_pose_t5.py'),
]
train_script = next((p for p in train_script_candidates if os.path.isfile(p)), None)
if train_script is None:
    raise FileNotFoundError({'searched': train_script_candidates})

env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([os.path.join(CODE, 'src'), CODE, env.get('PYTHONPATH', '')])
if os.path.isdir(MT5_LOCAL):
    env['TRANSFORMERS_OFFLINE'] = '1'
    env['HF_HUB_OFFLINE'] = '1'

phase1_cmd = [
    sys.executable, train_script,
    '--data-roots', MIXED_DATA,
    '--out-dir', PHASE1_DIR,
    '--base-model', base_model,
    *PHASE1_RESUME_ARGS,
    '--epochs', '200',
    '--batch-size', '4',
    '--grad-accum', '4',
    '--amp', 'auto',
    '--require-gpu', 'true',
    '--smoke-steps', '0',
    '--lr', '5e-5',
    '--dropout', '0.4',
    '--weight-decay', '0.1',
    '--num-encoder-layers', '2',
    '--downsample-factor', '4',
    '--balance-sources', 'auto',
    '--focus-target-tokens', 'ฉัน,คุณ,แม่,พี่,วันนี้,พรุ่งนี้',
    '--focus-target-max-multiplier', '3.0',
    '--preload-train-features', 'false',
    '--split-policy', 'manifest',
    '--required-sources', 'tsl51,thaisignvis,youtube_sl25_thai',
    '--manifest-quality-sources', 'tsl51',
    '--fail-on-manifest-quality', 'true',
    '--early-stopping-metric', 'val_loss',
    '--early-stopping-patience', '8',
    '--eval-steps', '200',
    '--checkpoint-steps', '5000',
    '--max-runtime-min', '420',
    '--reset-progress-history',
    '--evaluate-after-train', 'false',
    # mixed dataset has individual .npy files, not features.zip
    '--raw-required-files', 'manifest.csv,manifest_quality.json',
    '--staged-required-files', 'manifest.csv',
    # seed is at step 3075; smoke max_steps=20 < 3075 -> noop guard would fire without this
    '--allow-noop-resume', 'true',
]
print('[phase1] Starting pretrain on all 1938 examples...')
print('cmd:', ' '.join(str(x) for x in phase1_cmd))
result = subprocess.run(phase1_cmd, env=env)
if result.returncode != 0:
    # List phase1 dir so we can see any partial output
    import os as _os
    p1_files = sorted(_os.listdir(PHASE1_DIR)) if _os.path.isdir(PHASE1_DIR) else []
    raise RuntimeError(f'Phase 1 failed (rc={result.returncode}). phase1 files: {p1_files}')
print(f'[phase1] Completed successfully.')

In [ ]:
# === Handoff: copy Phase 1 best weights → Phase 2 seed ===
import os, shutil

src = os.path.join(PHASE1_DIR, 'best_model_state.pt')
dst = os.path.join(PHASE2_DIR, 'best_model_state.pt')
if not os.path.isfile(src):
    raise FileNotFoundError(f'Phase 1 did not produce best_model_state.pt at {src}')
if not os.path.isfile(dst):
    shutil.copy2(src, dst)
    print(f'[handoff] Copied {src} → {dst}')
else:
    print(f'[handoff] Phase 2 seed already present at {dst} (skipping copy)')

In [ ]:
# === Phase 2: finetune on TSL-51 only → export + eval ===
import os, subprocess, sys

phase2_cmd = [
    sys.executable, train_script,
    '--data-roots', TSL51_DATA,
    '--out-dir', PHASE2_DIR,
    '--base-model', base_model,
    '--resume', 'best_state',
    '--epochs', '200',
    '--batch-size', '4',
    '--grad-accum', '4',
    '--amp', 'auto',
    '--require-gpu', 'true',
    '--smoke-steps', '0',
    '--lr', '5e-6',
    '--dropout', '0.4',
    '--num-encoder-layers', '2',
    '--downsample-factor', '4',
    '--balance-sources', 'false',
    '--preload-train-features', 'false',
    '--split-policy', 'manifest',
    '--required-sources', 'tsl51',
    '--manifest-quality-sources', 'tsl51',
    '--fail-on-manifest-quality', 'false',
    '--early-stopping-metric', 'val_chrf',
    '--early-stopping-patience', '10',
    '--eval-steps', '25',
    '--checkpoint-steps', '5000',
    '--max-runtime-min', '200',
    '--reset-progress-history',
    # relax preflight for tsl51 (no features.zip / manifest_quality.json required)
    '--raw-required-files', 'manifest.csv',
    '--staged-required-files', 'manifest.csv',
    '--expected-manifest-rows', '0',
    '--expected-resolved-examples', '0',
    '--expected-source-counts', '',
    # export + eval inline (same protocol as incumbent: seed=42, n=25, tsl51)
    '--evaluate-after-train', 'true',
    '--eval-data-roots', TSL51_DATA,
    '--eval-report-data-roots', 'data/tsl51_v3',
    '--eval-sources', 'tsl51',
    '--eval-split-policy', 'manifest',
    '--eval-device', 'cpu',
    '--eval-val-subset-size', '25',
    '--min-val-chrf', '80',
    '--min-val-exact-match-pct', '60',
    # allow noop smoke step when resuming from Phase 1 checkpoint
    '--allow-noop-resume', 'true',
]
print('[phase2] Starting TSL-51 finetune...')
print('cmd:', ' '.join(phase2_cmd))
subprocess.run(phase2_cmd, check=True, env=env)

In [ ]:
# === Publish export as Kaggle dataset (best-effort) ===
import os, json, subprocess, sys
from pathlib import Path

EXPORT_FILES = [
    'config.json', 'generation_config.json', 'model.safetensors',
    'pose_encoder.pt', 'pose_t5_config.json', 'runtime_metadata.json',
    'tokenizer.json', 'tokenizer_config.json',
    'verified_eval.json', 'verified_samples.json', 'cloud_preflight.json',
]

phase2_files = sorted(os.listdir(PHASE2_DIR)) if os.path.isdir(PHASE2_DIR) else []
print(f'[publish] Phase2 output files ({len(phase2_files)}): {phase2_files}')

# Write dataset-metadata.json for orbitorls/thai-sign-tsl51-model
meta = {
    'title': 'Thai Sign TSL-51 Model',
    'id': 'orbitorls/thai-sign-tsl51-model',
    'licenses': [{'name': 'CC0-1.0'}],
}
meta_path = os.path.join(PHASE2_DIR, 'dataset-metadata.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

publish_script_candidates = [
    os.path.join(CODE, 'scripts', 'publish_kaggle_dataset.py'),
    os.path.join(CODE, 'publish_kaggle_dataset.py'),
]
publish_script = next((p for p in publish_script_candidates if os.path.isfile(p)), None)

if publish_script:
    try:
        publish_cmd = [
            sys.executable, publish_script,
            '--dataset-dir', PHASE2_DIR,
            '--message', 'curriculum one-shot tsl51-finetune',
        ]
        subprocess.run(publish_cmd, check=True, env=env, timeout=300)
        print('[publish] Dataset published to orbitorls/thai-sign-tsl51-model')
    except Exception as exc:
        print(f'[publish] WARNING: in-kernel publish failed ({exc}). Download via kaggle kernels output and publish locally.')
else:
    print('[publish] publish_kaggle_dataset.py not found — download output and publish locally.')

# Always list files so kaggle kernels output can retrieve them
print(f'\n[done] Files in {PHASE2_DIR}:')
for f in sorted(os.listdir(PHASE2_DIR)):
    size = os.path.getsize(os.path.join(PHASE2_DIR, f))
    print(f'  {f}  ({size/1024:.1f} KB)')